# Green Hill Pond — LIDAR Residential Density Analysis

**Water quality study:** Count residential structures near Green Hill Pond (South Kingstown / Charlestown, RI) as a proxy for septic/fertilizer groundwater loading.

**Data sources:**
- USGS LIDAR: `s3://usgs-lidar-public/RI_Statewide_1_D22/ept.json` (2022, 8 pts/m²)
- RIGIS parcels: RI statewide parcels (use codes 100–199 = residential)
- USGS WBD: HUC12 watershed boundaries

**Storage:** All data cached to Google Drive — survives between Colab sessions.

## 1. Setup — Mount Drive + Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# All persistent data goes here — survives between Colab sessions
DRIVE_BASE = Path('/content/drive/MyDrive/lidar-analysis')
CACHE_DIR  = DRIVE_BASE / 'cache'
OUTPUT_DIR = DRIVE_BASE / 'output'

for d in [CACHE_DIR, OUTPUT_DIR, CACHE_DIR / 'parcels']:
    d.mkdir(parents=True, exist_ok=True)

# Tell the lidar package where to cache
os.environ['LIDAR_CACHE_DIR'] = str(CACHE_DIR)

print(f'Cache:  {CACHE_DIR}')
print(f'Output: {OUTPUT_DIR}')
print()

# Show existing cached files
cached = list(CACHE_DIR.rglob('*')) + list(OUTPUT_DIR.rglob('*'))
if cached:
    print('Existing files on Drive:')
    for f in sorted(cached):
        if f.is_file():
            print(f'  {f.relative_to(DRIVE_BASE)}  ({f.stat().st_size / 1e6:.1f} MB)')
else:
    print('No cached files yet — first run will fetch from S3.')

In [ ]:
# Install system library (apt) + Python packages
# pdal has manylinux wheels on PyPI for Linux — no apt needed
# pysheds pinned <0.3 to avoid numba/llvmlite build issues

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

print('Installing dependencies...')
pip('pdal>=3.0')
pip('pysheds<0.3')
pip('scikit-learn>=1.3', 'scipy>=1.11')
pip('geopandas>=0.14', 'pyproj>=3.6', 'shapely>=2.0')
pip('rasterio>=1.3')
pip('folium>=0.15')
pip('requests>=2.31')

print('Done.')

# Verify pdal
import pdal
print(f'pdal {pdal.__version__} OK')

In [ ]:
# Install the lidar package from this repo
# Option A: if running from a Drive copy of the repo
#   %cd /content/drive/MyDrive/claudecode/projects/lidar && pip install -e .
#
# Option B: clone fresh from GitHub (update URL when repo is pushed)
#   !git clone https://github.com/YOUR_REPO/claudecode /content/claudecode
#   %pip install -e /content/claudecode/projects/lidar
#
# Option C (default): install inline from source cells below (no repo needed)
# — the key modules are included directly in this notebook as helper cells

print('Using inline module definitions (no repo clone needed).')
print('To use the full package instead, uncomment Option A or B above.')

## 2. Configuration

In [ ]:
# ── Study area ──────────────────────────────────────────────────────────────
POND_LAT    = 41.3707      # Green Hill Pond centroid
POND_LON    = -71.6156
FETCH_RADIUS_M = 1500      # Radius to fetch LIDAR (meters)
RADII       = [500, 1000]  # Buffer radii for residential counts

# ── Data sources ────────────────────────────────────────────────────────────
EPT_URL = 's3://usgs-lidar-public/RI_Statewide_1_D22/ept.json'

# ── Raster resolution ───────────────────────────────────────────────────────
# 0.3m = native LIDAR pulse spacing (most detail, ~1.2 GB rasters)
# 0.5m = good balance (~430 MB rasters, faster)
# 1.0m = fastest (~110 MB rasters, adequate for residential counting)
RESOLUTION = 0.5

# ── Query points ────────────────────────────────────────────────────────────
# 10 points around Green Hill Pond shoreline
QUERY_POINTS = [
    {'label': 'pond_center',     'lat': 41.3707, 'lon': -71.6156},
    {'label': 'north_shore',     'lat': 41.3770, 'lon': -71.6156},
    {'label': 'south_shore',     'lat': 41.3645, 'lon': -71.6156},
    {'label': 'east_shore',      'lat': 41.3707, 'lon': -71.6040},
    {'label': 'west_shore',      'lat': 41.3707, 'lon': -71.6270},
    {'label': 'nw_residential',  'lat': 41.3760, 'lon': -71.6240},
    {'label': 'sw_residential',  'lat': 41.3660, 'lon': -71.6230},
    {'label': 'inlet_north',     'lat': 41.3780, 'lon': -71.6110},
    {'label': 'outlet_south',    'lat': 41.3640, 'lon': -71.6090},
    {'label': 'east_residential','lat': 41.3720, 'lon': -71.5990},
]

print(f'Study area: {POND_LAT}°N, {POND_LON}°W  |  radius={FETCH_RADIUS_M}m  |  res={RESOLUTION}m')
print(f'Query points: {len(QUERY_POINTS)}')

## 3. Fetch LIDAR Point Cloud from USGS S3

In [ ]:
"""fetch.py — inline (mirrors src/lidar/fetch.py)"""
import hashlib, json, time
import numpy as np
from pyproj import Transformer

FT_TO_M = 0.3048006

def _bounds_3857(lat, lon, radius_m):
    t = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)
    cx, cy = t.transform(lon, lat)
    return (cx - radius_m, cy - radius_m, cx + radius_m, cy + radius_m)

def _fmt_bounds(b):
    return f'([{b[0]},{b[2]}],[{b[1]},{b[3]}])'

def fetch_point_cloud(lat, lon, radius_m=1500, ept_url=EPT_URL,
                      classes=(2,3,4,5,6), use_cache=True):
    import pdal
    bounds = _bounds_3857(lat, lon, radius_m)
    key = hashlib.md5(f'{bounds}{classes}'.encode()).hexdigest()
    cache_path = CACHE_DIR / f'{key}.npy'
    meta_path  = cache_path.with_suffix('.json')

    if use_cache and cache_path.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text())
        if time.time() - meta['cached_at'] < 4 * 3600:
            print(f'  Loaded from Drive cache: {cache_path.name}')
            return np.load(str(cache_path))

    class_filter = '|'.join(f'Classification[{c}:{c}]' for c in classes)
    pipeline_def = {'pipeline': [
        {'type': 'readers.ept', 'filename': ept_url, 'bounds': _fmt_bounds(bounds)},
        {'type': 'filters.range', 'limits': class_filter},
    ]}
    print('  Fetching from S3...')
    pipeline = pdal.Pipeline(json.dumps(pipeline_def))
    pipeline.execute()
    arr = pipeline.arrays[0].copy()
    arr['Z'] = arr['Z'] * FT_TO_M

    if use_cache:
        np.save(str(cache_path), arr)
        meta_path.write_text(json.dumps({'cached_at': time.time()}))
        print(f'  Cached to Drive: {cache_path.stat().st_size / 1e6:.0f} MB')
    return arr

def get_class(arr, cls):  return arr[arr['Classification'] == cls]
def get_classes(arr, *cls):
    mask = np.zeros(len(arr), dtype=bool)
    for c in cls: mask |= arr['Classification'] == c
    return arr[mask]

print('fetch helpers ready')

In [ ]:
print(f'Fetching LIDAR for {POND_LAT}°N, {POND_LON}°W  (radius={FETCH_RADIUS_M}m)...')
arr = fetch_point_cloud(POND_LAT, POND_LON, radius_m=FETCH_RADIUS_M, ept_url=EPT_URL)

print(f'\nTotal points: {len(arr):,}')
for cls, name in [(2,'Ground'), (3,'Low veg'), (4,'Med veg'), (5,'High veg'), (6,'Building')]:
    n = (arr['Classification'] == cls).sum()
    print(f'  Class {cls} ({name}): {n:,}  ({100*n/len(arr):.1f}%)')

class6 = get_class(arr, 6)
print(f'\nClass 6 (buildings): {len(class6):,} points')

## 4. Detect Buildings (DBSCAN on Class-6 Points)

In [ ]:
"""buildings.py — inline"""
import numpy as np
import geopandas as gpd
from shapely.geometry import MultiPoint, Point
from pyproj import Transformer
from sklearn.cluster import DBSCAN

def detect_buildings(class6_arr, input_crs='EPSG:3857',
                     eps=3.0, min_samples=10,
                     min_area_m2=10.0, max_area_m2=5000.0):
    if len(class6_arr) == 0:
        return gpd.GeoDataFrame(columns=['geometry','area_m2','point_count','cluster_id'], crs='EPSG:4326')

    if input_crs != 'EPSG:32619':
        t = Transformer.from_crs(input_crs, 'EPSG:32619', always_xy=True)
        xs, ys = t.transform(class6_arr['X'], class6_arr['Y'])
    else:
        xs, ys = class6_arr['X'], class6_arr['Y']

    coords = np.column_stack([xs, ys])
    db = DBSCAN(eps=eps, min_samples=min_samples, algorithm='ball_tree').fit(coords)
    labels = db.labels_

    records = []
    for label in set(labels):
        if label == -1: continue
        mask = labels == label
        hull = MultiPoint(coords[mask]).convex_hull
        area = hull.area
        if area < min_area_m2 or area > max_area_m2: continue
        c = hull.centroid
        records.append({'cluster_id': label, 'centroid_x_m': c.x,
                         'centroid_y_m': c.y, 'area_m2': area, 'point_count': int(mask.sum())})

    if not records:
        return gpd.GeoDataFrame(columns=['geometry','area_m2','point_count','cluster_id'], crs='EPSG:4326')

    gdf = gpd.GeoDataFrame(records,
        geometry=[Point(r['centroid_x_m'], r['centroid_y_m']) for r in records],
        crs='EPSG:32619')
    return gdf.to_crs('EPSG:4326')

print('buildings helpers ready')

In [ ]:
print('Running DBSCAN building detection...')
buildings = detect_buildings(class6)
print(f'Detected {len(buildings)} buildings')
if len(buildings):
    print(f'  Area range: {buildings.area_m2.min():.0f} – {buildings.area_m2.max():.0f} m²')
    print(f'  Median area: {buildings.area_m2.median():.0f} m²')
buildings.head()

## 5. Residential Classification (RIGIS Parcels)

In [ ]:
"""parcels.py — inline"""
import time, json
import geopandas as gpd
import requests

RIGIS_REST = 'https://gis.ri.gov/server/rest/services/Parcels/RI_Statewide_Parcels/MapServer/0/query'
USE_CODE_FIELDS = ['USE_CODE','PROP_CLASS','LAND_USE','USECD','USE_CD','USECODE']
_parcel_cache = CACHE_DIR / 'parcels' / 'green_hill_parcels.gpkg'

def _normalize_use_code(gdf):
    gdf = gdf.copy()
    for f in USE_CODE_FIELDS:
        if f in gdf.columns:
            gdf['use_code'] = gdf[f]
            break
    else:
        gdf['use_code'] = None
        return gdf
    gdf['use_code'] = gdf['use_code'].astype(str).str.extract(r'(\d+)')[0]
    gdf['use_code'] = gdf['use_code'].astype(float).astype('Int64')
    return gdf

def load_parcels(bbox, use_cache=True):
    if use_cache and _parcel_cache.exists():
        age_h = (time.time() - _parcel_cache.stat().st_mtime) / 3600
        if age_h < 7*24:
            print(f'  Loaded parcels from Drive cache ({age_h:.0f}h old)')
            return gpd.read_file(_parcel_cache)

    xmin, ymin, xmax, ymax = bbox
    params = {'geometry': f'{xmin},{ymin},{xmax},{ymax}',
              'geometryType': 'esriGeometryEnvelope', 'inSR': '4326',
              'spatialRel': 'esriSpatialRelIntersects',
              'outFields': '*', 'returnGeometry': 'true',
              'f': 'geojson', 'resultRecordCount': 2000}
    resp = requests.get(RIGIS_REST, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if not data.get('features'):
        print('  Warning: no parcels returned from RIGIS REST')
        return gpd.GeoDataFrame(columns=['geometry','use_code','is_residential'], crs='EPSG:4326')
    gdf = gpd.GeoDataFrame.from_features(data['features'], crs='EPSG:4326')
    gdf = _normalize_use_code(gdf)
    code = gdf['use_code']
    gdf['is_residential'] = (code >= 100) & (code <= 199)
    _parcel_cache.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(str(_parcel_cache), driver='GPKG')
    print(f'  Parcels cached to Drive ({len(gdf)} records)')
    return gdf

def join_buildings_parcels(buildings, parcels):
    if buildings.empty or parcels.empty:
        buildings = buildings.copy(); buildings['is_residential'] = None; return buildings
    if parcels.crs != buildings.crs: parcels = parcels.to_crs(buildings.crs)
    slim = parcels[['geometry','use_code','is_residential']].copy()
    joined = gpd.sjoin(buildings, slim, how='left', predicate='within')
    joined = joined[~joined.index.duplicated(keep='first')]
    buildings = buildings.copy()
    buildings['use_code'] = joined.get('use_code')
    buildings['is_residential'] = joined.get('is_residential')
    return buildings

print('parcels helpers ready')

In [ ]:
deg_per_m = 1 / 111_320
bbox = (
    POND_LON - FETCH_RADIUS_M * deg_per_m,
    POND_LAT - FETCH_RADIUS_M * deg_per_m,
    POND_LON + FETCH_RADIUS_M * deg_per_m,
    POND_LAT + FETCH_RADIUS_M * deg_per_m,
)

print('Loading RIGIS parcel data...')
try:
    parcels = load_parcels(bbox)
    buildings = join_buildings_parcels(buildings, parcels)
    n_res = buildings['is_residential'].sum()
    n_unk = buildings['is_residential'].isna().sum()
    print(f'  Residential: {n_res}  |  Non-res: {len(buildings)-n_res-n_unk}  |  Unknown: {n_unk}')
except Exception as e:
    print(f'  Warning: parcel load failed ({e}) — buildings will be unclassified')
    buildings['is_residential'] = None

## 6. HUC12 Watershed Boundary

In [ ]:
"""watershed.py — inline"""
import json, time
import geopandas as gpd
import requests

_huc_cache = CACHE_DIR / 'huc12.geojson'

def fetch_huc12(lat, lon, use_cache=True):
    if use_cache and _huc_cache.exists():
        age_h = (time.time() - _huc_cache.stat().st_mtime) / 3600
        if age_h < 7*24:
            print(f'  Loaded HUC12 from Drive cache ({age_h:.0f}h old)')
            return gpd.read_file(str(_huc_cache))

    url = 'https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/6/query'
    params = {'geometry': json.dumps({'x': lon, 'y': lat}),
              'geometryType': 'esriGeometryPoint', 'inSR': '4326',
              'spatialRel': 'esriSpatialRelIntersects',
              'outFields': 'huc12,name,areasqkm',
              'returnGeometry': 'true', 'outSR': '4326', 'f': 'geojson'}
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    features = data.get('features', [])
    if not features:
        return gpd.GeoDataFrame(columns=['geometry','huc12','name','areasqkm'], crs='EPSG:4326')
    gdf = gpd.GeoDataFrame.from_features(features, crs='EPSG:4326')
    if len(gdf) > 1:
        unioned = gdf.geometry.unary_union
        gdf = gpd.GeoDataFrame(
            [{'huc12': ', '.join(gdf['huc12']), 'name': ', '.join(gdf['name']),
              'areasqkm': gdf['areasqkm'].sum(), 'geometry': unioned}], crs='EPSG:4326')
    gdf.to_file(str(_huc_cache), driver='GeoJSON')
    print(f'  HUC12 cached to Drive')
    return gdf

print('watershed helpers ready')

In [ ]:
print('Fetching HUC12 watershed...')
watershed = None
try:
    watershed = fetch_huc12(POND_LAT, POND_LON)
    if not watershed.empty:
        row = watershed.iloc[0]
        print(f'  {row["name"]}  |  {row["areasqkm"]:.1f} km²  |  HUC12: {row["huc12"]}')
except Exception as e:
    print(f'  Warning: HUC12 fetch failed ({e})')

## 7. Terrain Rasters (DEM / DSM / CHM / Slope)

This takes 5–15 minutes on first run. Output is saved to Drive and reused on subsequent runs.

In [ ]:
"""terrain.py — inline (key functions)"""
import json
from pathlib import Path
import numpy as np
import rasterio
from rasterio.fill import fillnodata
from pyproj import Transformer

NODATA = -9999.0

def _bounds_str(lat, lon, radius_m):
    t = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)
    cx, cy = t.transform(lon, lat)
    xmin, ymin, xmax, ymax = cx-radius_m, cy-radius_m, cx+radius_m, cy+radius_m
    return f'([{xmin},{xmax}],[{ymin},{ymax}])'

def build_dem(lat, lon, radius_m, out_path, ept_url=EPT_URL, resolution=0.5):
    import pdal
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    pipeline = pdal.Pipeline(json.dumps({'pipeline': [
        {'type': 'readers.ept', 'filename': ept_url, 'bounds': _bounds_str(lat, lon, radius_m)},
        {'type': 'filters.range', 'limits': 'Classification[2:2]'},
        {'type': 'filters.reprojection', 'in_srs': 'EPSG:3857', 'out_srs': 'EPSG:32619'},
        {'type': 'filters.assign', 'assignment': 'Z[:]=Z*0.3048006'},
        {'type': 'writers.gdal', 'filename': str(out_path), 'resolution': resolution,
         'output_type': 'mean', 'data_type': 'float32', 'nodata': NODATA,
         'gdalopts': 'COMPRESS=LZW', 'override_srs': 'EPSG:32619'},
    ]}))
    pipeline.execute()
    return out_path

def build_dsm(lat, lon, radius_m, out_path, ept_url=EPT_URL, resolution=0.5):
    import pdal
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    pipeline = pdal.Pipeline(json.dumps({'pipeline': [
        {'type': 'readers.ept', 'filename': ept_url, 'bounds': _bounds_str(lat, lon, radius_m)},
        {'type': 'filters.range', 'limits': 'Classification[2:5]'},
        {'type': 'filters.reprojection', 'in_srs': 'EPSG:3857', 'out_srs': 'EPSG:32619'},
        {'type': 'filters.assign', 'assignment': 'Z[:]=Z*0.3048006'},
        {'type': 'writers.gdal', 'filename': str(out_path), 'resolution': resolution,
         'output_type': 'max', 'data_type': 'float32', 'nodata': NODATA,
         'gdalopts': 'COMPRESS=LZW', 'override_srs': 'EPSG:32619'},
    ]}))
    pipeline.execute()
    return out_path

def build_chm(dem_path, dsm_path, out_path):
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(dem_path) as d, rasterio.open(dsm_path) as s:
        dem = d.read(1).astype('float32')
        dsm = s.read(1).astype('float32')
        profile = d.profile.copy()
        valid = (dem != NODATA) & (dsm != NODATA)
        chm = np.where(valid, np.maximum(dsm - dem, 0.0), NODATA)
        profile.update(nodata=NODATA)
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(chm, 1)
    return out_path

def compute_slope(dem_path, out_path):
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(dem_path) as src:
        dem = src.read(1).astype('float32')
        res = src.res; profile = src.profile.copy(); nodata = src.nodata or NODATA
    valid = dem != nodata
    filled = dem.copy(); filled[~valid] = float(np.nanmean(dem[valid])) if valid.any() else 0.0
    dy, dx = np.gradient(filled, res[0], res[1])
    slope_pct = np.tan(np.arctan(np.sqrt(dx**2 + dy**2))) * 100.0
    slope_pct[~valid] = NODATA
    profile.update(dtype='float32', nodata=NODATA)
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(slope_pct.astype('float32'), 1)
    return out_path

def fill_nodata_raster(raster_path, max_search_distance=10):
    with rasterio.open(raster_path, 'r+') as src:
        data = src.read(1); nodata = src.nodata or NODATA
        mask = (data != nodata).astype('uint8')
        src.write(fillnodata(data, mask=mask, max_search_distance=max_search_distance), 1)

print('terrain helpers ready')

In [ ]:
dem_path   = OUTPUT_DIR / 'dem_bare_earth.tif'
dsm_path   = OUTPUT_DIR / 'dsm_with_trees.tif'
chm_path   = OUTPUT_DIR / 'chm.tif'
slope_path = OUTPUT_DIR / 'slope.tif'

if dem_path.exists():
    print('DEM already on Drive — skipping fetch.')
    print('Delete output/dem_bare_earth.tif from Drive to re-run.')
else:
    print(f'Building DEM (class 2, {RESOLUTION}m)...')
    build_dem(POND_LAT, POND_LON, FETCH_RADIUS_M, dem_path, resolution=RESOLUTION)
    fill_nodata_raster(dem_path)
    print(f'  DEM: {dem_path.stat().st_size/1e6:.0f} MB')

    print(f'Building DSM (class 2–5, {RESOLUTION}m)...')
    build_dsm(POND_LAT, POND_LON, FETCH_RADIUS_M, dsm_path, resolution=RESOLUTION)
    fill_nodata_raster(dsm_path)
    print(f'  DSM: {dsm_path.stat().st_size/1e6:.0f} MB')

    print('Building CHM...')
    build_chm(dem_path, dsm_path, chm_path)
    print(f'  CHM: {chm_path.stat().st_size/1e6:.0f} MB')

    print('Computing slope...')
    compute_slope(dem_path, slope_path)
    print(f'  Slope: {slope_path.stat().st_size/1e6:.0f} MB')
    print('Terrain rasters saved to Drive.')

## 8. Spatial Analysis — Residential Counts per Query Point

In [ ]:
"""spatial.py — inline"""
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
import rasterio
from rasterio.mask import mask as rio_mask

def count_in_radius(query_point, buildings_gdf, radius_m):
    if buildings_gdf.empty: return {'total': 0, 'residential': 0}
    t = Transformer.from_crs('EPSG:4326', 'EPSG:32619', always_xy=True)
    cx, cy = t.transform(query_point.x, query_point.y)
    buf = Point(cx, cy).buffer(radius_m)
    within = buildings_gdf.to_crs('EPSG:32619')[buildings_gdf.to_crs('EPSG:32619').geometry.within(buf)]
    total = len(within)
    res = int(within['is_residential'].fillna(False).sum()) if 'is_residential' in within.columns else 0
    return {'total': total, 'residential': res}

def elev_stats(dem_path, slope_path, lat, lon, radius_m):
    t = Transformer.from_crs('EPSG:4326', 'EPSG:32619', always_xy=True)
    cx, cy = t.transform(lon, lat)
    geom = [Point(cx, cy).buffer(radius_m).__geo_interface__]
    def _mean(path):
        try:
            with rasterio.open(path) as src:
                data, _ = rio_mask(src, geom, crop=True, nodata=NODATA)
            arr = data[0].astype('float32')
            v = arr[arr != NODATA]
            return float(np.nanmean(v)) if len(v) > 0 else float('nan')
        except: return float('nan')
    return {'mean_elev_m': round(_mean(dem_path), 2),
            'mean_slope_pct': round(_mean(slope_path), 2)}

def count_in_watershed(buildings_gdf, watershed_gdf):
    if buildings_gdf.empty or watershed_gdf is None or watershed_gdf.empty:
        return {'total': 0, 'residential': 0}
    ws = watershed_gdf.geometry.unary_union
    within = buildings_gdf[buildings_gdf.geometry.within(ws)]
    total = len(within)
    res = int(within['is_residential'].fillna(False).sum()) if 'is_residential' in within.columns else 0
    return {'total': total, 'residential': res}

print('spatial helpers ready')

In [ ]:
ws_counts = count_in_watershed(buildings, watershed)
has_terrain = dem_path.exists() and slope_path.exists()

records = []
for pt in QUERY_POINTS:
    geom = Point(pt['lon'], pt['lat'])
    rec  = {'label': pt['label'], 'lat': pt['lat'], 'lon': pt['lon']}
    for r in RADII:
        c = count_in_radius(geom, buildings, r)
        rec[f'res_{r}m']   = c['residential']
        rec[f'total_{r}m'] = c['total']
    rec['res_huc12']   = ws_counts['residential']
    rec['total_huc12'] = ws_counts['total']
    if has_terrain:
        stats = elev_stats(dem_path, slope_path, pt['lat'], pt['lon'], max(RADII))
        rec.update(stats)
    records.append(rec)

results = pd.DataFrame(records)

# Save to Drive
results_csv = OUTPUT_DIR / 'results.csv'
results.to_csv(str(results_csv), index=False)
print(f'Results saved to Drive: {results_csv}')

results

## 9. Interactive Map (Folium)

In [ ]:
import folium
from IPython.display import display

m = folium.Map(location=[POND_LAT, POND_LON], zoom_start=14, tiles='OpenStreetMap')

# HUC12 watershed
if watershed is not None and not watershed.empty:
    ws_layer = folium.FeatureGroup(name='HUC12 Watershed', show=True)
    folium.GeoJson(watershed.__geo_interface__,
        style_function=lambda _: {'fillColor':'none','color':'#2e7d32','weight':3,'dashArray':'8 4'},
        tooltip=watershed.iloc[0].get('name','HUC12')).add_to(ws_layer)
    ws_layer.add_to(m)

# Residential buildings
res_layer   = folium.FeatureGroup(name='Residential Buildings', show=True)
other_layer = folium.FeatureGroup(name='Other Buildings', show=False)
for _, b in buildings.iterrows():
    lat, lon = b.geometry.y, b.geometry.x
    is_res = b.get('is_residential')
    tip = f"{'Res' if is_res else 'Other'} | {b.get('area_m2',0):.0f} m²"
    if is_res:
        folium.CircleMarker([lat,lon],radius=4,color='#c62828',fill=True,
            fill_color='#ef5350',fill_opacity=0.7,tooltip=tip).add_to(res_layer)
    else:
        folium.CircleMarker([lat,lon],radius=3,color='#546e7a',fill=True,
            fill_color='#90a4ae',fill_opacity=0.5,tooltip=tip).add_to(other_layer)
res_layer.add_to(m)
other_layer.add_to(m)

# Query points with buffers
pts_layer = folium.FeatureGroup(name='Query Points', show=True)
for _, row in results.iterrows():
    lat, lon = row['lat'], row['lon']
    lines = [f"<b>{row['label']}</b>"]
    for r in RADII:
        lines.append(f"{r}m: {row.get(f'res_{r}m','?')} res / {row.get(f'total_{r}m','?')} total")
    lines.append(f"Watershed: {row.get('res_huc12','?')} res / {row.get('total_huc12','?')} total")
    if 'mean_elev_m' in row and not pd.isna(row['mean_elev_m']):
        lines.append(f"Elev: {row['mean_elev_m']:.1f}m  |  Slope: {row['mean_slope_pct']:.1f}%")
    for r, op in zip(sorted(RADII), [0.12, 0.06]):
        folium.Circle([lat,lon],radius=r,color='#e65100',fill=True,
            fill_color='#ff6d00',fill_opacity=op,weight=1).add_to(pts_layer)
    folium.Marker([lat,lon],
        popup=folium.Popup('<br>'.join(lines), max_width=250),
        tooltip=row['label'],
        icon=folium.Icon(color='blue', icon='info-sign')).add_to(pts_layer)
pts_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# Save to Drive
map_path = OUTPUT_DIR / 'green_hill_map.html'
m.save(str(map_path))
print(f'Map saved to Drive: {map_path}')

# Display inline
m

## 10. Export — Buildings GeoJSON + Results Summary

In [ ]:
# Buildings GeoJSON
bldg_path = OUTPUT_DIR / 'buildings.geojson'
buildings.to_file(str(bldg_path), driver='GeoJSON')
print(f'Buildings: {bldg_path}  ({len(buildings)} records)')

# Summary table
print('\n=== Results Summary ===')
print(results.to_string(index=False))

# Drive inventory
print('\n=== Drive files ===')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        print(f'  {f.name:<35} {f.stat().st_size/1e6:.1f} MB')
total = sum(f.stat().st_size for f in OUTPUT_DIR.rglob('*') if f.is_file())
total += sum(f.stat().st_size for f in CACHE_DIR.rglob('*') if f.is_file())
print(f'\nTotal Drive usage: {total/1e9:.2f} GB')